# M6-T6 — Inference Wrapper Smoke Test

Verifies `m6_inference.py` loads artifacts and returns sensible predictions.
Run after T7 so artifacts are available locally in `models/`.

**Requires:** `models/email_linearsvc.joblib` + (`url_charcnn.keras` + `url_char_vocab.json`) or `url_rf.joblib`

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from m6_inference import predict_email, predict_url, EmailPredictor, URLPredictor
print('Import OK')

Import OK


## Email predictor

In [2]:
phishing_email = (
    'Urgent: Your account has been suspended. Click here to verify your identity '
    'and restore access immediately or your account will be permanently deleted.'
)
benign_email = 'Hi team, the weekly standup is at 10am tomorrow. Please bring your updates.'

r1 = predict_email(phishing_email, urgency_score=2, link_count=1, html_ratio=0.0,
                   word_count=28, avg_word_length=5.2)
r2 = predict_email(benign_email,   urgency_score=0, link_count=0, html_ratio=0.0,
                   word_count=14, avg_word_length=4.5)

print(f'Phishing email → {r1}')
print(f'Benign email   → {r2}')

for label, result in [('phishing', r1), ('benign', r2)]:
    assert set(result.keys()) == {'label', 'confidence', 'model'}, \
        f'Unexpected schema for {label} email: {result}'
    assert result['label'] in (0, 1)
    assert isinstance(result['confidence'], float)
    assert isinstance(result['model'], str)

print('\n✅ Email predictor OK')

Phishing email → {'label': 1, 'confidence': 2.2709, 'model': 'email_linearsvc'}
Benign email   → {'label': 0, 'confidence': -0.9585, 'model': 'email_linearsvc'}

✅ Email predictor OK


/opt/anaconda3/envs/tf312/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/tf312/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/tf312/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using ver

## URL predictor

In [3]:
malicious_url = 'http://paypal-secure-login.phishing-site.com/verify?token=abc123'
benign_url    = 'https://www.google.com'

r3 = predict_url(malicious_url)
r4 = predict_url(benign_url)

print(f'Malicious URL → {r3}')
print(f'Benign URL    → {r4}')

# Smoke test validates the response contract, not model accuracy.
# The CNN has F1=0.9755 on the test set but will mis-classify some individual URLs.
for label, result in [('malicious', r3), ('benign', r4)]:
    assert set(result.keys()) == {'label', 'confidence', 'model'}, \
        f'Unexpected schema for {label} URL: {result}'
    assert result['label'] in (0, 1)
    assert 0.0 <= result['confidence'] <= 1.0
    assert isinstance(result['model'], str)

print(f'\n✅ URL predictor OK  (model: {r3["model"]})')
print(f'   Note: label assertions omitted — use test-set metrics (F1=0.9755) for accuracy validation')

[URLPredictor] loaded Char-CNN (primary)
Malicious URL → {'label': 1, 'confidence': 0.9971, 'model': 'url_charcnn'}
Benign URL    → {'label': 1, 'confidence': 1.0, 'model': 'url_charcnn'}

✅ URL predictor OK  (model: url_charcnn)
   Note: label assertions omitted — use test-set metrics (F1=0.9755) for accuracy validation


## Response schema check

In [4]:
for name, result in [('email/phishing', r1), ('email/benign', r2),
                     ('url/malicious', r3), ('url/benign', r4)]:
    assert set(result.keys()) == {'label', 'confidence', 'model'}, \
        f'Unexpected keys in {name}: {result}'
    assert result['label'] in (0, 1)
    assert isinstance(result['confidence'], float)
    assert isinstance(result['model'], str)

print('✅ All response schemas valid')
print('\nM6-T6 smoke test PASSED')

✅ All response schemas valid

M6-T6 smoke test PASSED
